# PE6201 A2 — V2 Live Evaluation

This notebook runs the complete V2 claim-assessment agent with `openai/gpt-4o-mini`, including tool calls, guardrails, deterministic checks and an independent judgement check.

## 1. Runtime and project setup

In [1]:
from pathlib import Path
from datetime import datetime
from IPython.display import display, JSON
import csv
import hashlib
import json
import os
import shlex
import subprocess
import sys
import time

SEARCH_ROOTS = [Path.cwd(), Path.cwd().parent, Path('/content/PE6201_A2_Final_Deliverables')]
ROOT = next((p.resolve() for p in SEARCH_ROOTS if (p / 'code' / 'V2' / 'run_eval.py').exists()), None)
assert ROOT is not None, 'Open this notebook from the PE6201_A2_Final_Deliverables folder.'
CODE = ROOT / 'code' / 'V2'
AGENT_MODEL = 'openai/gpt-4o-mini'
PROMPT_VERSION = 'v2'
JUDGE_MODEL = 'google/gemini-2.5-flash-lite'
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = ROOT / 'live_run_outputs' / f'v2_gpt4o_mini_{RUN_TAG}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

print('Project root :', ROOT)
print('V2 code      :', CODE)
print('Python       :', sys.executable)
print('Output folder:', OUTPUT_DIR)

Project root : /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables
V2 code      : /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/code/V2
Python       : /Users/wendys/anaconda3/bin/python
Output folder: /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/live_run_outputs/v2_gpt4o_mini_20260920_032724


In [2]:
def dotenv_has_key(path: Path, key: str) -> bool:
    if not path.exists():
        return False
    for raw in path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if line and not line.startswith('#') and line.startswith(key + '='):
            return bool(line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39)))
    return False

key_ready = bool(os.getenv('OPENROUTER_API_KEY')) or dotenv_has_key(CODE / '.env', 'OPENROUTER_API_KEY')
required = ['run_eval.py', 'agent.py', 'tools.py', 'guardrails.py', 'harness.py', 'backends.py']
missing = [name for name in required if not (CODE / name).exists()]
assert not missing, f'Missing V2 files: {missing}'
assert key_ready, 'Set OPENROUTER_API_KEY in the environment or code/V2/.env.'

print('V2 source files:', len(required), '/', len(required))
print('API key configured:', key_ready)
print('Evaluated agent model :', AGENT_MODEL)
print('Agent prompt version  :', PROMPT_VERSION)
print('Independent evaluator :', JUDGE_MODEL, '(judge only; not the evaluated agent)')

V2 source files: 6 / 6
API key configured: True
Evaluated agent model : openai/gpt-4o-mini
Agent prompt version  : v2
Independent evaluator : google/gemini-2.5-flash-lite (judge only; not the evaluated agent)


## 2. Selected cases(including negative)

`CLM-8910` and `CLM-8917` are two policy-boundary negative cases. Each is evaluated three times according to the evaluation protocol. `CLM-9023` is an ordinary inclusive-boundary case evaluated once.

In [3]:
claims = {row['claim_id']: row for row in json.loads((CODE / 'data_A' / 'claims.json').read_text(encoding='utf-8'))}
expected = {row['case_id']: row for row in json.loads((CODE / 'expected_outcomes_A.json').read_text(encoding='utf-8'))}
negative_cases = ['CLM-8910', 'CLM-8917']
ordinary_cases = ['CLM-9023']
selected_cases = negative_cases + ordinary_cases

for case_id in selected_cases:
    key = expected[case_id]
    print('-' * 80)
    print('Case             :', case_id)
    print('Family           :', key.get('family'))
    print('Expected decision:', key.get('expected_decision'))
    print('Negative case    :', key.get('expected_decision') != 'approve_in_principle')
    print('Claim lines      :', len(claims[case_id].get('lines', [])))
    print('Must record      :', len(key.get('must_record', [])), 'items')

--------------------------------------------------------------------------------
Case             : CLM-8910
Family           : policy_lapsed
Expected decision: escalate
Negative case    : True
Claim lines      : 3
Must record      : 2 items
--------------------------------------------------------------------------------
Case             : CLM-8917
Family           : outside_policy_dates
Expected decision: escalate
Negative case    : True
Claim lines      : 1
Must record      : 2 items
--------------------------------------------------------------------------------
Case             : CLM-9023
Family           : policy_start_date_inclusive
Expected decision: approve_in_principle
Negative case    : False
Claim lines      : 1
Must record      : 4 items


## 3. Live process runner

In [4]:
def stream_process(args):
    env = dict(os.environ)
    env['PYTHONUNBUFFERED'] = '1'
    printable = ' '.join(shlex.quote(str(item)) for item in args)
    print('$', printable, flush=True)
    started = time.time()
    process = subprocess.Popen(
        [str(item) for item in args],
        cwd=CODE,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    elapsed = time.time() - started
    print(f'\nProcess finished: exit={return_code}, elapsed={elapsed:.1f}s', flush=True)
    if return_code != 0:
        raise RuntimeError(f'Live evaluation exited with code {return_code}')
    return elapsed

COMMON_ARGS = [
    sys.executable, CODE / 'run_eval.py',
    '--backend', 'live',
    '--model', AGENT_MODEL,
    '--model-family', 'OpenAI GPT-4o mini',
    '--prompt-version', PROMPT_VERSION,
    '--judge', 'live',
    '--judge-model', JUDGE_MODEL,
    '--verbose',
    '--output-dir', OUTPUT_DIR,
]
print('Live runner ready.')

Live runner ready.


In [6]:
raw_json = OUTPUT_DIR / 'v2_live_raw_results.json'
raw_csv = OUTPUT_DIR / 'v2_live_raw_results.csv'
monitor_command = (
    f"while [ ! -f {shlex.quote(str(raw_csv))} ]; do sleep 1; done; "
    f"tail -f {shlex.quote(str(raw_csv))}"
)
print('Terminal monitor command:')
print(monitor_command)

Terminal monitor command:
while [ ! -f /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/live_run_outputs/v2_gpt4o_mini_20260920_032724/v2_live_raw_results.csv ]; do sleep 1; done; tail -f /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/live_run_outputs/v2_gpt4o_mini_20260920_032724/v2_live_raw_results.csv


## 4. Run two negative cases

The first case tests a lapsed policy. The second tests a service date outside the policy period. Both require a structured escalation record with the correct single trigger.

In [7]:
negative_elapsed = 0.0
for index, case_id in enumerate(negative_cases):
    case_args = COMMON_ARGS + ['--case', case_id, '--max-runs', '3']
    if index > 0:
        case_args += ['--resume']
    negative_elapsed += stream_process(case_args)

assert raw_json.exists() and raw_csv.exists()
negative_payload = json.loads(raw_json.read_text(encoding='utf-8'))
negative_runs = negative_payload['raw_runs']
assert len(negative_runs) == 6
assert {row['case_id'] for row in negative_runs} == set(negative_cases)
assert all(row['negative_case'] for row in negative_runs)
assert all(sum(row['case_id'] == case_id for row in negative_runs) == 3 for case_id in negative_cases)
print('Saved negative-case runs:', len(negative_runs))
print('Negative cases          :', ', '.join(negative_cases))
print('JSON:', raw_json)
print('CSV :', raw_csv)

$ /Users/wendys/anaconda3/bin/python /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/code/V2/run_eval.py --backend live --model openai/gpt-4o-mini --model-family 'OpenAI GPT-4o mini' --prompt-version v2 --judge live --judge-model google/gemini-2.5-flash-lite --verbose --output-dir /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/live_run_outputs/v2_gpt4o_mini_20260920_032724 --case CLM-8910 --max-runs 3
BACKEND=live | PROBLEM=A | provider=openrouter | model=openai/gpt-4o-mini | prompt=v2 | temperature=0 | max_output_tokens=1200 | reasoning=disabled | cap=8 turns | autonomy=confirm
data=/Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/code/V2
planned new runs=3; existing preserved=0
  turn 1    · I need to fetch the claim details for case CLM-8910 to proceed.
       get_claim                  args={'claim_id': 'CLM-8910'} -> {'claim_id': 'CLM-8910', 'member_id': 'M-4471', 'hospital_id': 'H-114', 'date_of_service': '202…
  turn 2    · I have the claim de

In [8]:
negative_table = []
for row in negative_runs:
    negative_table.append({
        'case_id': row['case_id'],
        'trial': row['trial'],
        'expected': row['expected_decision'],
        'actual': row['actual_decision'],
        'turns': row['turns'],
        'tools_called': len(row['tools_called']),
        'code_pass': row['code_check_pass'],
        'judge_pass': row['judgement_check_pass'],
        'strict_pass': row['overall_pass'],
        'stopped_by': row['stopped_by'],
    })
display(JSON(negative_table, expanded=True))

print('Strict pass rule: code_pass AND judge_pass')
strict_passes = sum(row['overall_pass'] is True for row in negative_runs)
print(f'Negative strict passes: {strict_passes}/{len(negative_runs)}')
failed_runs = [row for row in negative_runs if row['overall_pass'] is not True]
for row in failed_runs:
    reason = ' | '.join(row.get('code_check_failures') or [])
    if not reason:
        reason = row.get('judgement', {}).get('reason', 'No failure reason recorded')
    print(f"FAIL detail: {row['case_id']} trial {row['trial']} -> {reason}")

<IPython.core.display.JSON object>

Strict pass rule: code_pass AND judge_pass
Negative strict passes: 6/6


## 5. Add one ordinary boundary case to the same result files

In [9]:
ordinary_elapsed = stream_process(
    COMMON_ARGS + [
        '--case', 'CLM-9023',
        '--max-runs', '1',
        '--resume',
    ]
)
payload = json.loads(raw_json.read_text(encoding='utf-8'))
runs = payload['raw_runs']
assert len(runs) == 7
assert sum(row['negative_case'] for row in runs) == 6
assert {row['case_id'] for row in runs} == set(selected_cases)
print('Total saved runs:', len(runs))
print('Negative trials :', sum(row['negative_case'] for row in runs))
print('Ordinary runs   :', sum(not row['negative_case'] for row in runs))

$ /Users/wendys/anaconda3/bin/python /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/code/V2/run_eval.py --backend live --model openai/gpt-4o-mini --model-family 'OpenAI GPT-4o mini' --prompt-version v2 --judge live --judge-model google/gemini-2.5-flash-lite --verbose --output-dir /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/live_run_outputs/v2_gpt4o_mini_20260920_032724 --case CLM-9023 --max-runs 1 --resume
BACKEND=live | PROBLEM=A | provider=openrouter | model=openai/gpt-4o-mini | prompt=v2 | temperature=0 | max_output_tokens=1200 | reasoning=disabled | cap=8 turns | autonomy=confirm
data=/Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/code/V2
planned new runs=1; existing preserved=6
  turn 1    · I need to fetch the claim details for case CLM-9023 to proceed.
       get_claim                  args={'claim_id': 'CLM-9023'} -> {'claim_id': 'CLM-9023', 'member_id': 'M-7023', 'hospital_id': 'H-207', 'date_of_service': '202…
  turn 2    · Now that I

## 6. Inspect the newly generated evidence

In [10]:
chosen = next(row for row in runs if row['case_id'] == 'CLM-8910' and row['trial'] == 1)
print('Run ID         :', chosen['run_id'])
print('Expected       :', chosen['expected_decision'])
print('Actual         :', chosen['actual_decision'])
print('Turns          :', chosen['turns'])
print('Strict pass    :', chosen['overall_pass'])
print('Stopped by     :', chosen['stopped_by'])
print('Tools called   :')
for index, tool_name in enumerate(chosen['tools_called'], start=1):
    print(f'  {index:02d}. {tool_name}')

print('\nFinal decision record:')
display(JSON(chosen['record'], expanded=False))

Run ID         : v2-CLM-8910-t1-9ffddb41
Expected       : escalate
Actual         : escalate
Turns          : 2
Strict pass    : True
Stopped by     : final_answer
Tools called   :
  01. get_claim
  02. lookup_policy
  03. lookup_hospital

Final decision record:


<IPython.core.display.JSON object>

In [11]:
evidence = {
    'model': payload['config']['model'],
    'prompt_version': payload['config']['prompt_version'],
    'judge_model': payload['config']['judge_model'],
    'raw_result_json': str(raw_json),
    'runs': [],
}
for row in runs:
    evidence['runs'].append({
        'run_id': row['run_id'],
        'case_id': row['case_id'],
        'trial': row['trial'],
        'negative_case': row['negative_case'],
        'expected_decision': row['expected_decision'],
        'actual_decision': row['actual_decision'],
        'turns': row['turns'],
        'tools_called': row['tools_called'],
        'code_check_pass': row['code_check_pass'],
        'judgement_check_pass': row['judgement_check_pass'],
        'overall_pass': row['overall_pass'],
        'stopped_by': row['stopped_by'],
        'decision_record': row['record'],
    })

evidence_path = OUTPUT_DIR / 'v2_gpt4o_run_evidence.json'
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding='utf-8')
print('Evidence file:', evidence_path)
print('Size         :', evidence_path.stat().st_size, 'bytes')

Evidence file: /Users/wendys/Documents/AI/NTU/PE6201_A2_Final_Deliverables/live_run_outputs/v2_gpt4o_mini_20260920_032724/v2_gpt4o_run_evidence.json
Size         : 59100 bytes


In [12]:
def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

with raw_csv.open(encoding='utf-8-sig', newline='') as handle:
    csv_rows = list(csv.DictReader(handle))
assert len(csv_rows) == len(runs) == 7
assert json.loads(evidence_path.read_text(encoding='utf-8'))['runs'][0]['run_id'] == runs[0]['run_id']

for path in [raw_json, raw_csv, evidence_path]:
    print(f'{path.name:32} {path.stat().st_size:>9,} bytes  sha256={sha256(path)[:16]}...')
print('\nAll result-file checks passed.')

v2_live_raw_results.json            94,057 bytes  sha256=4dacf0e13f3af797...
v2_live_raw_results.csv             20,340 bytes  sha256=3092f49f7a3e61ba...
v2_gpt4o_run_evidence.json          59,100 bytes  sha256=41496714a467b4f4...

All result-file checks passed.
